# PubMed RAG: Chunking Strategy Pipeline

## Settings

In [1]:
import sys
import csv
import os
import json
import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load .env
load_dotenv(find_dotenv())

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = os.environ.get("DATA_DIR", "/workspace/data")

PARSED_DIR = os.path.join(DATA_DIR, "parsed_docs")
CHUNK_DIR = os.path.join(DATA_DIR, "chunked_docs")
train_csv_path = os.path.join(PARSED_DIR, "training_final.csv")
test_csv_path = os.path.join(PARSED_DIR, "test_final.csv")

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")
print(f"Train CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

SEED: 42
CUDA available: True
PyTorch version: 2.2.0
Data dir: /workspace/data
Train CSV path: /workspace/data/parsed_docs/training_final.csv
Test CSV path: /workspace/data/parsed_docs/test_final.csv


## Helpers

In [2]:
from ftfy import fix_text
import unicodedata
import html
import re
import pandas as pd

def clean_pubmed_text(text):

    if pd.isna(text):
        return ""

    # fix encoding problems
    text = fix_text(text)

    # decode html entities
    text = html.unescape(text)

    # remove html tags
    text = re.sub(r"<.*?>", " ", text)

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # normalize weird unicode spaces
    text = re.sub(r"[\u2000-\u200F\u202F\u205F]", " ", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Load Data

In [3]:
df = pd.read_csv(train_csv_path)

dup = df["pmid"].duplicated().sum()
print("Duplicate PMIDs:", dup)

df = df.drop_duplicates(
    subset=["pmid"]
).reset_index(drop=True)

print("Unique PMIDs:", df["pmid"].nunique())
print("Total rows:", len(df))


Duplicate PMIDs: 1847
Unique PMIDs: 37595
Total rows: 37595


## Chunking Experimental Setup

| Strategy                        | Description                                                                                                           | Typical Chunk Size      |
| ------------------------------- | --------------------------------------------------------------------------------------------------------------------- | ----------------------- |
| **section_contextual_400token** | Section-aware chunks enriched with LLM-generated summaries. Improves semantic representation and retrieval alignment. | ~400 tokens (+ context) |
| **section_400token**            | Section-aware chunks (~400 tokens). Preserves semantic unit boundaries.                                               | ~400 tokens             |
| **fixed_500char**               | Fixed-length chunks (500 characters). May split mid-sentence.                                                         | ~500 characters         |


In [4]:
!pip install transformers accelerate torch

### Strategy 1 — Contextual Chunking

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    return_full_text=False
)

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
def batch_summarize(sections, batch_size=64):
    
    summaries = []
    
    for i in range(0, len(sections), batch_size):
        
        batch = sections[i:i+batch_size]

        prompts = []

        for title, label, text in batch:

            prompt = f"""
You are assisting a biomedical retrieval system.

Write ONE concise sentence (maximum 30 words) describing the main biomedical topic and research focus of the text.

Do NOT repeat the text. Provide only the summary sentence.

Title: {title}
Section: {label}

Text:
{text}

Summary:
"""
            prompts.append(prompt)

        results = llm(prompts)

        for r in results:

            if isinstance(r, list):
                generated = r[0]["generated_text"]
            else:
                generated = r["generated_text"]

            summary = generated.split("Summary:")[-1].strip()
            summary = summary.replace("\n", " ")

            summaries.append(summary)

    return summaries

In [7]:
import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ======================================================
# Start timer
# ======================================================

start_time = time.time()

# ======================================================
# Tokenizer (used only for token length estimation)
# ======================================================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ======================================================
# Token-based text splitter
# ======================================================

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=60,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)

# ======================================================
# Store generated chunks
# ======================================================

chunks = []

# ======================================================
# Main processing loop
# ======================================================

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # ------------------------------------------------
    # TITLE chunk
    # ------------------------------------------------

    if title:
        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })

    # ------------------------------------------------
    # Parse abstract sections
    # ------------------------------------------------

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        continue

    # ------------------------------------------------
    # Process each section
    # ------------------------------------------------

    for sec in sections:

        label = sec.get("label") or "ABSTRACT"
        text = clean_pubmed_text(sec.get("text", ""))

        if not text:
            continue

        # ------------------------------------------------
        # Extract first sentence as lightweight context
        # ------------------------------------------------

        first_sentence = text.split(". ")[0].strip()

        # Prevent extremely long first sentence
        first_sentence = " ".join(first_sentence.split()[:30])

        context_prefix = f"""
        Title: {title}
        Section: {label}
        Context: {first_sentence}
        """

        # ------------------------------------------------
        # Compute token length
        # ------------------------------------------------

        token_len = tokenizer(
            text,
            add_special_tokens=False,
            return_length=True
        )["length"][0]

        # ------------------------------------------------
        # SHORT SECTIONS
        # ------------------------------------------------

        if token_len <= 400:

            contextual_text = f"{context_prefix}\n{text}"

            chunks.append({
                "pmid": pmid,
                "topic_id": topic_id,
                "category": category,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "title": title,
                "chunk_text": contextual_text
            })

        # ------------------------------------------------
        # LONG SECTIONS
        # ------------------------------------------------

        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                contextual_text = f"{context_prefix}\n{chunk}"

                chunks.append({
                    "pmid": pmid,
                    "topic_id": topic_id,
                    "category": category,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "title": title,
                    "chunk_text": contextual_text
                })

# ======================================================
# Convert to DataFrame
# ======================================================

contextual_chunks_df = pd.DataFrame(chunks)

# ======================================================
# Remove duplicates
# ======================================================

before = len(contextual_chunks_df)

contextual_chunks_df = contextual_chunks_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(contextual_chunks_df)

print(f"Removed {before - after} duplicate chunks")

# ======================================================
# Save dataset
# ======================================================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_section_contextual_400token.csv"

contextual_chunks_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# ======================================================
# Runtime statistics
# ======================================================

end_time = time.time()

print("\nTotal chunks:", len(contextual_chunks_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(contextual_chunks_df.head())

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Processing articles: 100% 37595/37595 [00:55<00:00, 678.30it/s]


Removed 7 duplicate chunks

Total chunks: 109234
Runtime: 58.31 seconds

Sample output:
       pmid  topic_id       category   section             chunk_id  \
0  15829955         8  Rare Diseases     TITLE       15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_ABSTRACT_0   
2  15617541         8  Rare Diseases     TITLE       15617541_TITLE   
3  15617541         8  Rare Diseases  ABSTRACT  15617541_ABSTRACT_0   
4  12239580         8  Rare Diseases     TITLE       12239580_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  Studying the genetics of Hirschsprung's diseas...   
3  Studying the genetics of Hirschsprung's diseas...   
4  Hirschsprung, RET-SOX and beyond: the challeng...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  \n        Title: A common sex-depen

In [13]:
# character length
contextual_chunks_df["char_len"] = contextual_chunks_df["chunk_text"].str.len()

# word count
contextual_chunks_df["word_count"] =contextual_chunks_df["chunk_text"].str.split().str.len()

print("Character length distribution:")
print(contextual_chunks_df["char_len"].describe())

print("\nWord count distribution:")
print(contextual_chunks_df["word_count"].describe())

Character length distribution:
count    109234.000000
mean        743.580259
std         626.485491
min           7.000000
25%         121.000000
50%         643.000000
75%        1178.000000
max        3668.000000
Name: char_len, dtype: float64

Word count distribution:
count    109234.000000
mean        101.681830
std          88.021632
min           1.000000
25%          16.000000
50%          85.000000
75%         164.000000
max         509.000000
Name: word_count, dtype: float64


### Strategy 2 — Section-Aware Chunking

In [8]:
import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ======================================================
# Start timer to measure total runtime
# ======================================================

start_time = time.time()

# ======================================================
# Tokenizer (used for token counting during chunking)
# ======================================================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ======================================================
# Token-based recursive text splitter
# Splits long sections into chunks while attempting
# to preserve paragraph/sentence boundaries
# ======================================================

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,      # maximum tokens per chunk
    chunk_overlap=60,    # overlapping tokens between chunks
    separators=[
        "\n\n",          # prefer paragraph splits
        "\n",            # then line breaks
        ". ",            # finally sentence boundaries
    ]
)

# ======================================================
# Store all generated chunks
# ======================================================

chunks = []

# ======================================================
# Main loop: iterate through each article
# ======================================================

for _, row in tqdm(df.iterrows(), total=len(df)):

    # --------------------------------------------------
    # Extract metadata
    # --------------------------------------------------

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # ==================================================
    # TITLE CHUNK
    # Create a separate chunk for the article title
    # This improves retrieval when queries match titles
    # ==================================================

    if title:
        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })

    # ==================================================
    # ABSTRACT SECTIONS
    # Parse structured abstract sections stored as JSON
    # ==================================================

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        continue

    # ==================================================
    # Process each section in the abstract
    # ==================================================

    for sec in sections:

        label = sec.get("label") or "ABSTRACT"
        text = clean_pubmed_text(sec.get("text", ""))

        # skip empty sections
        if not text:
            continue

        # ------------------------------------------------
        # Calculate token length of section
        # ------------------------------------------------

        token_len = len(tokenizer(text)["input_ids"])

        # ==================================================
        # SHORT SECTION
        # If section ≤ 400 tokens, keep it as one chunk
        # ==================================================

        if token_len <= 400:

            chunks.append({
                "pmid": pmid,
                "topic_id": topic_id,
                "category": category,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "title": title,
                "chunk_text": text
            })

        # ==================================================
        # LONG SECTION
        # Split section into multiple overlapping chunks
        # ==================================================

        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                chunks.append({
                    "pmid": pmid,
                    "topic_id": topic_id,
                    "category": category,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "title": title,
                    "chunk_text": chunk
                })


# ======================================================
# Convert list of chunks to DataFrame
# ======================================================

section_chunks_df = pd.DataFrame(chunks)

# ======================================================
# Remove duplicate chunks (based on chunk_id)
# ======================================================

before = len(section_chunks_df)

section_chunks_df = section_chunks_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(section_chunks_df)

print(f"Removed {before - after} duplicate chunks")

# ======================================================
# Save chunk dataset to disk
# ======================================================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_section_400token.csv"

section_chunks_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# ======================================================
# Runtime statistics
# ======================================================

end_time = time.time()

print("\nTotal chunks:", len(section_chunks_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(section_chunks_df.head())

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100% 37595/37595 [00:53<00:00, 707.50it/s]


Removed 7 duplicate chunks

Total chunks: 109234
Runtime: 55.7 seconds

Sample output:
       pmid  topic_id       category   section             chunk_id  \
0  15829955         8  Rare Diseases     TITLE       15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_ABSTRACT_0   
2  15617541         8  Rare Diseases     TITLE       15617541_TITLE   
3  15617541         8  Rare Diseases  ABSTRACT  15617541_ABSTRACT_0   
4  12239580         8  Rare Diseases     TITLE       12239580_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  Studying the genetics of Hirschsprung's diseas...   
3  Studying the genetics of Hirschsprung's diseas...   
4  Hirschsprung, RET-SOX and beyond: the challeng...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  The identification of common variant

In [9]:
'''import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()

# =========================
# Tokenizer (for token chunking)
# =========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=60,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)

# =========================
# Chunking
# =========================

chunks = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # =========================
    # TITLE CHUNK
    # =========================

    if title:
        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })

    # =========================
    # ABSTRACT SECTIONS
    # =========================

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        continue

    for sec in sections:

        label = sec.get("label") or "ABSTRACT"
        text = clean_pubmed_text(sec.get("text", ""))

        if not text:
            continue

        token_len = len(tokenizer(text)["input_ids"])

        # short section
        if token_len <= 400:

            chunks.append({
                "pmid": pmid,
                "topic_id": topic_id,
                "category": category,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "title": title,
                "chunk_text": text
            })

        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                chunks.append({
                    "pmid": pmid,
                    "topic_id": topic_id,
                    "category": category,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "title": title,
                    "chunk_text": chunk
                })


# =========================
# Convert to DataFrame
# =========================

chunk_df = pd.DataFrame(chunks)

# =========================
# Deduplicate chunks
# =========================

before = len(chunk_df)

chunk_df = chunk_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(chunk_df)

print(f"Removed {before - after} duplicate chunks")

# =========================
# Save
# =========================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_section_400token.csv"

chunk_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

end_time = time.time()

print("\nTotal chunks:", len(chunk_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(chunk_df.head())'''

'import pandas as pd\nimport json\nimport re\nimport html\nimport unicodedata\nimport time\nfrom tqdm import tqdm\nfrom pathlib import Path\n\nfrom transformers import AutoTokenizer\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\n\nstart_time = time.time()\n\n# =========================\n# Tokenizer (for token chunking)\n# =========================\n\nmodel_name = "Qwen/Qwen2.5-1.5B-Instruct"\ntokenizer = AutoTokenizer.from_pretrained(model_name)\n\nsplitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(\n    tokenizer,\n    chunk_size=400,\n    chunk_overlap=60,\n    separators=[\n        "\n\n",\n        "\n",\n        ". ",\n    ]\n)\n\n# =========================\n# Chunking\n# =========================\n\nchunks = []\n\nfor _, row in tqdm(df.iterrows(), total=len(df)):\n\n    pmid = str(row["pmid"])\n    title = clean_pubmed_text(row.get("title", ""))\n\n    topic_id = row.get("topic_id")\n    category = row.get("category")\n\n    # ============

In [10]:
import re

def find_special_chars(text):
    return re.findall(r"[^\x00-\x7F]", str(text))

section_chunks_df["weird_chars"] = section_chunks_df["chunk_text"].apply(find_special_chars)

char_counts = (
    section_chunks_df["weird_chars"]
    .explode()
    .value_counts()
)

print(char_counts.head(30))

weird_chars
β    4541
α    3994
·    3082
±    2361
≥    1578
μ    1165
γ     923
κ     787
®     622
×     597
≤     495
Δ     468
é     358
∼     347
‐     327
δ     288
ï     269
°     210
ö     168
©     164
•     115
ε     108
€     105
−      93
→      88
ç      88
è      84
ü      79
á      71
Å      63
Name: count, dtype: int64


In [11]:
# character length
section_chunks_df["char_len"] = section_chunks_df["chunk_text"].str.len()

# word count
section_chunks_df["word_count"] =section_chunks_df["chunk_text"].str.split().str.len()

print("Character length distribution:")
print(section_chunks_df["char_len"].describe())

print("\nWord count distribution:")
print(section_chunks_df["word_count"].describe())

Character length distribution:
count    109234.000000
mean        530.838613
std         519.236889
min           2.000000
25%         113.000000
50%         308.000000
75%         858.000000
max        3210.000000
Name: char_len, dtype: float64

Word count distribution:
count    109234.000000
mean         76.171998
std          75.258706
min           1.000000
25%          15.000000
50%          44.000000
75%         125.000000
max         458.000000
Name: word_count, dtype: float64


### Strategy 3 — Fixed Length Chunk (>500)

In [14]:
import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ======================================================
# Start timer to measure total runtime
# ======================================================

start_time = time.time()


# =========================
# Fixed-length splitter (500 chars)
# This splitter breaks text into chunks of 500 characters
# with an overlap of 80 characters between chunks.
# separators=[""] forces pure fixed-length splitting
# rather than semantic splitting by sentences or paragraphs.
# =========================

splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=[""]
)


# =========================
# Chunking
# Store all generated chunks here
# =========================

fixed_chunks = []

# Iterate through each row (article) in the dataset
for _, row in tqdm(df.iterrows(), total=len(df)):

    # --------------------------------------------------
    # Extract metadata for the article
    # --------------------------------------------------

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # =========================
    # TITLE CHUNK
    # Store the article title as its own chunk.
    # This can improve retrieval when queries match titles.
    # =========================

    if title:
        fixed_chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })


    # =========================
    # ABSTRACT SECTIONS
    # Parse the JSON containing abstract sections
    # =========================

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        # Skip rows where JSON parsing fails
        continue


    # Join all abstract sections into one continuous text
    # This removes section boundaries so the entire abstract
    # is chunked as a single sequence.
    full_text = " ".join(
        clean_pubmed_text(sec.get("text", ""))
        for sec in sections
    )


    # Skip if abstract text is empty
    if not full_text:
        continue


    # =========================
    # Fixed-length chunking
    # Split the full abstract text into 500-character chunks
    # =========================

    split_texts = splitter_500.split_text(full_text)

    for i, chunk in enumerate(split_texts):

        fixed_chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "ABSTRACT",
            "chunk_id": f"{pmid}_F500_{i}",
            "title": title,
            "chunk_text": chunk
        })

# =========================
# Convert chunk list to DataFrame
# =========================

fixed_df = pd.DataFrame(fixed_chunks)

# =========================
# Deduplicate chunks
# Remove duplicate entries based on chunk_id
# =========================

before = len(fixed_df)

fixed_df = fixed_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(fixed_df)

print(f"Removed {before - after} duplicate chunks")

# =========================
# Save chunks to CSV file
# =========================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_fixed500char.csv"

fixed_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# ======================================================
# Print runtime statistics
# ======================================================

end_time = time.time()

print("\nTotal chunks:", len(fixed_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(fixed_df.head())

100% 37595/37595 [01:26<00:00, 435.58it/s]


Removed 0 duplicate chunks

Total chunks: 176626
Runtime: 89.17 seconds

Sample output:
       pmid  topic_id       category   section         chunk_id  \
0  15829955         8  Rare Diseases     TITLE   15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_0   
2  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_1   
3  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_2   
4  15617541         8  Rare Diseases     TITLE   15617541_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  A common sex-dependent mutation in a RET enhan...   
3  A common sex-dependent mutation in a RET enhan...   
4  Studying the genetics of Hirschsprung's diseas...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  The identification of common variants that con...  
2  dise

In [15]:
# character length
fixed_df["char_len"] = fixed_df["chunk_text"].str.len()

# word count
fixed_df["word_count"] = fixed_df["chunk_text"].str.split().str.len()

print("Character length distribution:")
print(fixed_df["char_len"].describe())

print("\nWord count distribution:")
print(fixed_df["word_count"].describe())

Character length distribution:
count    176626.000000
mean        370.675495
std         173.176073
min           7.000000
25%         172.000000
50%         499.000000
75%         500.000000
max         500.000000
Name: char_len, dtype: float64

Word count distribution:
count    176626.000000
mean         53.644305
std          26.168395
min           1.000000
25%          24.000000
50%          67.000000
75%          74.000000
max         119.000000
Name: word_count, dtype: float64
